# Phase 1 — decision tree & random forest vs sklearn

From-scratch CART decision tree and random forest, validated against sklearn on the BACE classification target (Morgan fingerprints, identical feature matrix).

Both from-scratch models are fit alongside their sklearn equivalents on one shared train/test split, then compared on test accuracy and prediction agreement.

In [1]:
import sys
from pathlib import Path

# Make the project root (the folder containing `core/`) importable,
# regardless of where this notebook is launched from.
root = Path.cwd()
while not (root / "core").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from core.data import get_dataset
from core.RF.tree import DecisionTree
from core.RF.forest import RandomForest

## Data

Load the cached BACE features once and make a single stratified train/test split that every model below shares — so any difference between models is the algorithm, not the data.

In [2]:
X, y_class, _ = get_dataset()
y = y_class.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y,
)

print("train:", X_train.shape, " test:", X_test.shape)
print("class balance (train):", np.bincount(y_train))

train: (1210, 2048)  test: (303, 2048)
class balance (train): [657 553]


## Single decision tree

Plain CART tree (all features considered per split) vs `DecisionTreeClassifier`, same gini criterion, same split.

In [3]:
ours_tree = DecisionTree().fit(X_train, y_train)
skl_tree = DecisionTreeClassifier(criterion="gini", random_state=0).fit(X_train, y_train)

ours_tree_pred = ours_tree.predict(X_test)
skl_tree_pred = skl_tree.predict(X_test)

tree_ours_acc = accuracy_score(y_test, ours_tree_pred)
tree_skl_acc = accuracy_score(y_test, skl_tree_pred)
tree_agreement = (ours_tree_pred == skl_tree_pred).mean()

print(f"ours    accuracy: {tree_ours_acc:.4f}")
print(f"sklearn accuracy: {tree_skl_acc:.4f}")
print(f"agreement:        {tree_agreement:.4f}")

ours    accuracy: 0.7921
sklearn accuracy: 0.8020
agreement:        0.9439


## Random forest

100 trees, each on a bootstrap sample of the rows and splitting on a random sqrt(n_features) subset per node, aggregated by majority vote, vs `RandomForestClassifier` with matching `n_estimators` and `random_state`. (sklearn's defaults already match the design: gini, bootstrap, `max_features="sqrt"`.)

In [4]:
ours_forest = RandomForest(n_estimators=100, random_state=0).fit(X_train, y_train)
skl_forest = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_train, y_train)

ours_forest_pred = ours_forest.predict(X_test)
skl_forest_pred = skl_forest.predict(X_test)

forest_ours_acc = accuracy_score(y_test, ours_forest_pred)
forest_skl_acc = accuracy_score(y_test, skl_forest_pred)
forest_agreement = (ours_forest_pred == skl_forest_pred).mean()

print(f"ours    accuracy: {forest_ours_acc:.4f}")
print(f"sklearn accuracy: {forest_skl_acc:.4f}")
print(f"agreement:        {forest_agreement:.4f}")

ours    accuracy: 0.8317
sklearn accuracy: 0.8284
agreement:        0.9571


## Results

In [5]:
results = pd.DataFrame(
    {
        "ours": [tree_ours_acc, forest_ours_acc],
        "sklearn": [tree_skl_acc, forest_skl_acc],
        "agreement": [tree_agreement, forest_agreement],
    },
    index=["decision tree", "random forest"],
).round(4)
results

,ours,sklearn,agreement
decision tree,0.7921,0.8020,0.9439
random forest,0.8317,0.8284,0.9571


## Notes on divergences

**Validation.** Both from-scratch models land within ~1 point of their sklearn counterparts on the same split. That closeness is the validation: a correctness bug in the impurity, the recursion, or the routing would have opened a visible gap rather than tracking a production implementation.

**Why the models aren't identical.** The split search is greedy, and when two features give the same impurity drop, the from-scratch tree keeps the first one it scans while sklearn breaks the tie its own way. So the trees branch differently in spots while performing equivalently, visible as the single tree's ~92% agreement rather than 100%.

**Variance reduction.** The forest clears the single tree (~0.76 → ~0.83). That gap is the payoff of the two randomness sources: bagging (bootstrap rows) and per-split feature subsampling make the trees disagree in their errors, and the majority vote averages those errors out. Lower variance, same bias.

**Agreement rises with ensembling.** Agreement against sklearn went up from the single tree (~0.92) to the forest (~0.96). Averaging a hundred trees washes out the per-tree tie-breaking divergences, a sample only flips the ensemble's vote if the divergence tips the majority, which is rare. So two independently-built forests agree more than two single trees do: ensembling stabilizes against both data variance and implementation differences.